# 14 Train/Tune CybSAGE CAGE-4 Policy

Run this notebook after the previous numbered stage completes.


In [1]:
from pathlib import Path
import os, sys

REPO_ROOT_OVERRIDE = None


def _looks_like_repo(base: Path) -> bool:
    return (base / "pyproject.toml").exists() and (base / "src" / "cybsage_marl").exists()


def _mount_drive_if_colab():
    if not Path("/content").exists():
        return
    if Path("/content/drive/MyDrive").exists() or Path("/content/drive/My Drive").exists():
        return
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception as exc:
        print("Drive auto-mount failed:", exc)


def _find_repo_root():
    _mount_drive_if_colab()
    candidates = []
    if REPO_ROOT_OVERRIDE:
        candidates.append(Path(REPO_ROOT_OVERRIDE).expanduser())
    if os.environ.get("CYBSAGE_REPO_ROOT"):
        candidates.append(Path(os.environ["CYBSAGE_REPO_ROOT"]).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.extend([
        Path("/content/drive/MyDrive/projects/implementation/IMPLEMENTATION"),
        Path("/content/drive/My Drive/projects/implementation/IMPLEMENTATION"),
        Path("/content/drive/MyDrive/IMPLEMENTATION"),
        Path("/content/drive/My Drive/IMPLEMENTATION"),
        Path("/content/drive/MyDrive/projects/implementation/CybSage-MARL[DeepakSir]/IMPLEMENTATION"),
        Path("/content/drive/My Drive/projects/implementation/CybSage-MARL[DeepakSir]/IMPLEMENTATION"),
        Path("/Volumes/LocalDrive1/onlyscholar-projects/CybSage-MARL[DeepakSir]/IMPLEMENTATION"),
    ])

    checked = []
    seen = set()
    for base in candidates:
        base = base.expanduser()
        if str(base) in seen:
            continue
        seen.add(str(base))
        checked.append(base)
        if _looks_like_repo(base):
            return base

    # Last-resort bounded search. This catches Drive shortcuts/copies while avoiding a full Drive crawl.
    search_roots = [Path("/content/drive/MyDrive"), Path("/content/drive/My Drive"), Path("/content")]
    for root in search_roots:
        if not root.exists():
            continue
        try:
            for pyproject in root.glob("**/pyproject.toml"):
                base = pyproject.parent
                checked.append(base)
                if base.name == "IMPLEMENTATION" and _looks_like_repo(base):
                    return base
        except Exception as exc:
            checked.append(Path(f"{root} [search failed: {exc}]"))

    raise RuntimeError(
        "Could not find CybSAGE IMPLEMENTATION repo root visible to this runtime.\n\n"
        "Most common fix: in Colab, run drive.mount('/content/drive') and confirm the folder exists under "
        "/content/drive/MyDrive/projects/implementation/IMPLEMENTATION.\n\n"
        "Checked:\n" + "\n".join(map(str, checked[-80:]))
    )


REPO_ROOT = _find_repo_root()
os.chdir(REPO_ROOT)
src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print("Repository root:", REPO_ROOT)


Repository root: /content/drive/MyDrive/projects/implementation/IMPLEMENTATION


In [2]:
!PYTHONPATH=src python scripts/run_stage.py cage4_cybsage --cage4-episodes 2 --cage4-max-steps 100 --cage4-seeds 42 43 44 45 46

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.13/dist-packages/pkg_resources/__init__.py:3148: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`

In [3]:
import json, pandas as pd
from pathlib import Path
summary = json.loads(Path("outputs/14_cybsage_cage4_policy/cybsage_cage4_summary.json").read_text())
print(summary)
profile = Path("outputs/14_cybsage_cage4_policy/cybsage_profile_summary.csv")
pd.read_csv(profile) if profile.exists() and profile.stat().st_size else "No profile table produced"

{'status': 'completed', 'selected_profile': 'decoy_context_mid', 'train_seeds': [42, 43], 'eval_seeds': [44, 45, 46], 'episodes': 6, 'max_steps': 100, 'reward_mean': -63.5, 'reward_std': 18.854707634964804}


,profile,total_reward,selection_priority
0,decoy_context_mid,-48.00,5.5
1,decoy_context_high,-48.00,5.2
2,decoy_context_low,-48.00,5.0
3,decoy_remove_pressure,-48.00,4.0
4,decoy_guarded,-48.00,3.0
5,deception,-93.50,1.0
6,balanced,-179.25,1.0
7,containment,-179.25,1.0
8,resilience,-552.00,1.0
